## Day 4 content

In [ ]:
import json
import csv
import io
from logging import config
from pathlib import Path

from jupyterlab.extensions import entry
from notebooks.Day04_File_IO_JSON import parsed2, pretty, loaded, parse_llm_json_response, semicolon_csv, tsv_data, \
    no_header_csv, sample, dialect, csv_row

from fastapi_app.routers import customers
from modules.day04_file_io_json import content

try:
    BASE_DIR = Path(__file__).parent.parent
except NameError:
    BASE_DIR = Path.cwd()

PROMPT_DIR = BASE_DIR / 'prompts'

print('BASE_DIR  :', BASE_DIR)
print('PROMPT_DIR:', PROMPT_DIR)


In [ ]:
#json.loads() -- str -> dict
import json

raw = '{"category": "TRACK_ORDER","confidence": "high"}'

print('Before:', type(raw), raw)
parsed = json.loads(raw)
print('After:', type(parsed), parsed)
print('Access:', parsed['category'])

In [ ]:
# json.dumps() -- dict --> str

config = {
    'model'         : 'gpt-4o',
    'max_tokens'    : 1024,
    'temperature'   : 0.2
}

compact = json.dumps(config)
pretty  = json.dumps(config, indent=2)

print("compact:", compact)
print("Type:", type(compact))
print()
print("pretty :")
print(pretty)

In [ ]:
with open('model_config.json','w',encoding='utf-8') as f:
    json.dump(config,f,indent=2)

In [ ]:
with open('model_config.json','r', encoding='utf-8') as f:
    loaded = json.load(f)

print('Loaded back  :', loaded)
print("Type         :", type(loaded).__name__)

In [ ]:
def load_system_prompt(filepath = 'sample_prompt.txt'):
    with open(filepath, 'r', encoding= 'utf-8') as f:
        content = f.read()

    return  content.strip()

In [ ]:
with open('sample_prompt.txt', 'w', encoding= 'utf-8') as f:
    f.write("## Role\n")
    f.write('You are a customer support assistant for ShopSmart.\n')
    f.write('\n')
    f.write('## Task\n')
    f.write('Answer using ONLY the information provided.')

In [ ]:
prompt = load_system_prompt('sample_prompt.txt')
print(f"Loaded {len(prompt)} characters:")
print(prompt)

In [ ]:
def save_llm_response_log(query,response,log_file= 'llm_calls.log'):

    with open(log_file,'a', encoding='utf-8') as f:
        entry = json.dumps({
            'query'     : query,
            'response'  : response
        })
        f.write(entry + '\n')

In [ ]:
save_llm_response_log('Where is my order?','Your order ORD-3042 is In Transit.')

In [ ]:
save_llm_response_log('Can I return this?', 'Returns accepted within 30 days.')

In [ ]:
print('Log Contents:')
with open('llm_calls.log','r', encoding='utf-8') as f :
    for i, line in enumerate(f):
        record = json.loads(line.strip())
        print(f"    [{i+1}] Q: {record['query']}")
        print(f"        A:  {record['response']}")

In [ ]:
def parse_llm_json_response(raw_response):
    cleaned = raw_response.strip()

    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n',1)[-1]
        cleaned = cleaned.rsplit('```',1)[0].split()

    return json.loads(cleaned)

In [ ]:
raw = '{"category": "TRACK_ORDER", "confidence": "high"}'
print('Before Parse -- type:', type(raw).__name__)
parsed = parse_llm_json_response(raw)
print('After parse  -- type:', type(parsed).__name__)
print('category  :', parsed['category'])
print('confidence:', parsed['confidence'])

In [ ]:
fenced = '```json\n{"intent": "CANCEL", "urgency": "high"}\n```'

print('Raw response from LLM:')
print(fenced)
print()


parsed2 = parse_llm_json_response(fenced)
print('After parse:', parsed2)
print('intent    :', parsed2['intent'])

In [4]:
import csv

with open('employee.csv', 'r', encoding='utf-8') as f:

    reader = csv.DictReader(f)
    customers = list(reader)
    print(customers)

[{'customer_id': '1001', 'name': 'Yogavardhan Reddy G', 'email': 'yogavardhanreddy@example.com,yogavardhanreddy@example2.com', 'city': 'Bengaluru', 'state': 'Ka'}, {'customer_id': '1002', 'name': 'Srinivasulu Reddy', 'email': 'srinu@example.com', 'city': 'Bangalore', 'state': 'KA', None: ['TS']}, {'customer_id': '1003', 'name': 'Rohith Sharma', 'email': 'rohith@example.com', 'city': 'Mumbai', 'state': 'MH'}]


In [12]:
for c in customers:
    print(f"  {c['customer_id']}    {c['name']:20s}     {c['city']}, {c['state']}")

print()
print('--- CSV type trap ---')
print('customer_id type     :',type(customers[0]['customer_id']).__name__, '<- str1')
print("String compare trap : '1001' > '200' =", '1001' > '200', '<- WRONG (alphabetic)')
print('Correct int compare : 1001  > 200  =',   1001  > 200)
print('Fix                 :', int (customers[0]['customer_id']))

  1001    Yogavardhan Reddy G      Bengaluru, Ka
  1002    Srinivasulu Reddy        Bangalore, KA
  1003    Rohith Sharma            Mumbai, MH

--- CSV type trap ---
customer_id type     : str <- str1
String compare trap : '1001' > '200' = False <- WRONG (alphabetic)
Correct int compare : 1001  > 200  = True
Fix                 : 1001


In [15]:
# semicolon seperated
import io

semicolon_csv = """customer_id;name;city
1001;Yogavardhan Reddy G;Bengaluru, Ka
1002;Srinivasulu Reddy;Bangalore, KA
"""

reader = csv.DictReader(io.StringIO(semicolon_csv),delimiter=';')
for row in reader:
    print(row)

{'customer_id': '1001', 'name': 'Yogavardhan Reddy G', 'city': 'Bengaluru, Ka'}
{'customer_id': '1002', 'name': 'Srinivasulu Reddy', 'city': 'Bangalore, KA'}


In [19]:
tsv_data = 'customer_id\tname\tcity\n1001\tYogavardhan Reddy G\tBengaluru, Ka\n1002\tSrinivasulu Reddy\tBangalore, KA'

reader = csv.DictReader(io.StringIO(tsv_data),delimiter='\t')
for row in reader:
    print(row)

{'customer_id': '1001', 'name': 'Yogavardhan Reddy G', 'city': 'Bengaluru, Ka'}
{'customer_id': '1002', 'name': 'Srinivasulu Reddy', 'city': 'Bangalore, KA'}


In [20]:
pipe_csv = """customer_id|name|city
1001|Yogavardhan Reddy G|Bengaluru, Ka
1002|Srinivasulu Reddy|Bangalore, KA"""

reader = csv.DictReader(io.StringIO(pipe_csv), delimiter='|')
for row in reader:
    print(row)

{'customer_id': '1001', 'name': 'Yogavardhan Reddy G', 'city': 'Bengaluru, Ka'}
{'customer_id': '1002', 'name': 'Srinivasulu Reddy', 'city': 'Bangalore, KA'}


In [21]:
no_header_csv = """1001,Yogavardhan Reddy G,Bengaluru,
1002,Srinivasulu Reddy,Bengaluru
"""
reader = csv.DictReader(
    io.StringIO(no_header_csv),
    fieldnames=['customer_id','name','city']
)
for row in reader:
    print(row)

{'customer_id': '1001', 'name': 'Yogavardhan Reddy G', 'city': 'Bengaluru', None: ['']}
{'customer_id': '1002', 'name': 'Srinivasulu Reddy', 'city': 'Bengaluru'}


In [24]:
sample = """customer_id,name,email,city,state
1001,Yogavardhan Reddy G,"yogavardhanreddy@example.com,yogavardhanreddy@example2.com",Bengaluru,Ka
1002,Srinivasulu Reddy,srinu@example.com,Bangalore,KA
1003,Rohith Sharma,rohith@example.com,Mumbai,MH
"""

dialect = csv.Sniffer().sniff(sample)
print(type(dialect))
print('Deleted delimeter:', repr(dialect.delimiter))
print("Quote character:", repr(dialect.quotechar))

<class 'type'>
Deleted delimeter: ','
Quote character: '"'


In [25]:
reader = csv.DictReader(io.StringIO(sample), dialect=dialect)
for row in reader:
    print(row)

{'customer_id': '1001', 'name': 'Yogavardhan Reddy G', 'email': 'yogavardhanreddy@example.com,yogavardhanreddy@example2.com', 'city': 'Bengaluru', 'state': 'Ka'}
{'customer_id': '1002', 'name': 'Srinivasulu Reddy', 'email': 'srinu@example.com', 'city': 'Bangalore', 'state': 'KA'}
{'customer_id': '1003', 'name': 'Rohith Sharma', 'email': 'rohith@example.com', 'city': 'Mumbai', 'state': 'MH'}


In [ ]:
system_prompt = load_system_prompt('sample_prompt.txt')
print(f'Loaded {len(system_prompt)} chars')
print(system_prompt)

In [30]:
csv_row     = {'customer_id': '1001', 'name': 'Yogavardhan Reddy G','city': 'Bengaluru'}
customer_id = int(csv_row['customer_id'])
customer_name = csv_row['name']

print(f"customer : {customer_name}  id={customer_id} type = {type(customer_id).__name__}")

customer : Yogavardhan Reddy G  id=1001 type = int


In [31]:
user_message = f'Customer: {customer_name}\nQuery: Where is my order #3042?'

print(user_message)

Customer: Yogavardhan Reddy G
Query: Where is my order #3042?


In [32]:
mock_llm_output = '```json\n{"category": "TRACK_ORDER", "reply": "Your order ORD-3042 is In Transit.", "confidence": "high"}\n```'

print('Raw LLM output:')
print(mock_llm_output)

Raw LLM output:
```json
{"category": "TRACK_ORDER", "reply": "Your order ORD-3042 is In Transit.", "confidence": "high"}
```


In [33]:
parsed     = parse_llm_json_response(mock_llm_output)
category   = parsed.get('category',   'UNKNOWN')
reply      = parsed.get('reply',       'No reply')
confidence = parsed.get('confidence',  'unknown')

print('Parsed dict :', parsed)
print('category    :', category)
print('reply       :', reply)
print('confidence  :', confidence)

NameError: name 'parse_llm_json_response' is not defined

In [34]:
save_llm_response_log(user_message, reply)

# Read back to confirm it was written
with open('llm_calls.log', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f'llm_calls.log now has {len(lines)} entries')
print('Last entry:', lines[-1].strip())


NameError: name 'save_llm_response_log' is not defined